In [2]:
print(123)

123


In [3]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [4]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [5]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [6]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [10]:
rec = answers[0]
rec

{'question': 'I just found this course — can I still join it now, or is it too late?',
 'answer_llm': 'Yes — you can still join it now.  \nIf you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [17]:
len(answers)

425

In [9]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [11]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: it says the course can still be joined and that obtaining a certificate requires submitting the project while submissions are open. This is semantically equivalent.', score='good')

In [12]:
calc_price(usage)

{'input_cost': 0.00022349999999999998,
 'output_cost': 0.000252,
 'total_cost': 0.0004755}

In [13]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [14]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: it says the course can still be joined, and that a certificate requires submitting the project before submissions close. This is semantically equivalent and not missing any important detail.', score='good')

In [15]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [18]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/425 [00:00<?, ?it/s]

In [19]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [20]:
df_eval = pd.DataFrame(evaluations)

In [23]:
df_eval.head()

,question,document,score,reasoning
0,I just found this course — can I still join it...,74eb249bbf,good,The AI answer preserves the key meaning of the...
1,"If I enroll late, will I still be able to get ...",74eb249bbf,good,The AI answer preserves the key point: late en...
2,Do I have to submit the project before submiss...,74eb249bbf,good,The AI answer preserves the key point from the...
3,Is it okay to start the course after it has al...,74eb249bbf,good,The AI answer captures the main point that it ...
4,What’s the deadline for project submission if ...,74eb249bbf,good,The AI answer captures the key point that the ...


In [24]:
df_eval.score.value_counts()

score
good    407
bad      18
Name: count, dtype: int64

In [25]:
df_eval.score.value_counts(normalize=True)

score
good    0.957647
bad     0.042353
Name: proportion, dtype: float64

In [21]:
calc_total_price(usages)

0.27886875000000033

In [22]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 407/425 = 95.76%


In [26]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
29,Is certificate eligibility tied to submitting ...,69d122f12e,bad,The AI answer reverses the main condition. The...
43,When should I expect the course to run again?,bd31146b0e,bad,The ground truth states a specific expected ru...
47,"Do the lesson pages have the video links, or d...",31456f4b5f,bad,The AI answer fails to provide the ground trut...
57,How do I know when a live session is happening...,d65e05bd7a,bad,The AI answer captures the key point that live...
87,What’s the minimum amount I can add to my Open...,554d0eb78b,bad,The AI answer fails to provide the key informa...


In [27]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

In [31]:
print(123)

123
